In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "../Data"
MODEL_DIR = BASE_DIR / "app" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)
print(DATA_DIR.exists())
print(MODEL_DIR)

/Users/m.mughees/Desktop/Pi-515-2026/src
True
/Users/m.mughees/Desktop/Pi-515-2026/src/app/models


In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (2099831, 26)
Test shape: (10000, 26)


,num__latitude,num__longitude,num__soil_temperature_level_1,num__total_precipitation,num__runoff,num__total_evaporation,num__potential_evaporation,num__2m_dewpoint_temperature,num__2m_temperature,num__snow_cover,...,cat__DistrictNa_Central,cat__DistrictNa_East Central,cat__DistrictNa_North Central,cat__DistrictNa_Northeast,cat__DistrictNa_Northwest,cat__DistrictNa_South Central,cat__DistrictNa_Southeast,cat__DistrictNa_Southwest,cat__DistrictNa_West Central,Soil_Moisture
0,-0.776986,1.426464,-1.369474,-0.367620,0.264376,1.097594,1.129800,-0.989945,-1.215082,0.768468,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.360234
1,-1.216539,0.035936,1.056937,-0.359515,0.063897,-0.850514,-0.224441,0.796616,0.823547,-0.269393,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.213469
2,-1.028159,1.205244,0.334309,-0.336933,-0.268701,-0.342088,0.007690,0.505021,0.467160,-0.269393,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.166913
3,0.478879,-0.912151,-0.674495,-0.368519,-0.352383,1.054375,0.972629,-0.400612,-0.537826,-0.265868,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.199984
4,-0.400226,0.035936,0.985257,-0.078794,0.318792,-1.182331,-0.441741,1.085233,0.926438,-0.269393,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.214700


In [6]:
features = [
    "num__latitude",
    "num__longitude",
    "num__total_precipitation",
    "num__runoff",
    "num__total_evaporation",
    "num__potential_evaporation",
    "num__2m_dewpoint_temperature",
    "num__2m_temperature",
    "num__year",
    "num__month",
    "num__day",
    "cat__DistrictNa_Central",
    "cat__DistrictNa_East Central",
    "cat__DistrictNa_North Central",
    "cat__DistrictNa_Northeast",
    "cat__DistrictNa_Northwest",
    "cat__DistrictNa_South Central",
    "cat__DistrictNa_Southeast",
    "cat__DistrictNa_Southwest",
    "cat__DistrictNa_West Central",
]

target = "num__soil_temperature_level_1"

missing = [col for col in features + [target] if col not in train_df.columns]
print("Missing columns:", missing)

Missing columns: []


In [7]:
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print(X_train.shape, X_test.shape)

(2099831, 20) (10000, 20)


In [8]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(name, y_true, y_pred):
    metrics = {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }
    print(f"\n{name}")
    print("-" * 30)
    for k, v in metrics.items():
        print(f"{k.upper()}: {v:.4f}")
    return metrics

In [9]:
soil_temp_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

soil_temp_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=12, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=200, n_jobs=-1, random_state=42)

In [10]:
train_pred = soil_temp_model.predict(X_train)
test_pred = soil_temp_model.predict(X_test)

train_metrics = evaluate("Soil Temp Train", y_train, train_pred)
test_metrics = evaluate("Soil Temp Test", y_test, test_pred)


Soil Temp Train
------------------------------
RMSE: 0.1533
MAE: 0.1121
R2: 0.9765

Soil Temp Test
------------------------------
RMSE: 0.1549
MAE: 0.1134
R2: 0.9767


In [11]:
results_df = pd.DataFrame({
    "actual_soil_temp": y_test.values,
    "predicted_soil_temp": test_pred
})

results_df.head(10)

,actual_soil_temp,predicted_soil_temp
0,0.566021,0.546704
1,0.523393,0.324663
2,-1.119452,-1.108653
3,-1.264064,-1.385890
4,-1.571763,-1.642091
5,-1.708761,-1.703268
6,-0.029814,0.059963
7,-0.936033,-1.055243
8,-1.546663,-1.392080
9,-1.541344,-1.449043


In [12]:
joblib.dump(soil_temp_model, MODEL_DIR / "soil_temperature_model.joblib")

with open(MODEL_DIR / "soil_temp_metrics.json", "w") as f:
    json.dump(
        {
            "train": train_metrics,
            "test": test_metrics,
        },
        f,
        indent=2,
    )

results_df.head(20).to_csv(MODEL_DIR / "soil_temp_predictions.csv", index=False)

print("Saved model and outputs.")

Saved model and outputs.
